In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('HI-Small_Trans.csv')

df.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


# Data Cleaning

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 426.2 MB


In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.info()

<class 'pandas.DataFrame'>
Index: 5078336 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 464.9 MB


In [7]:
print(df.isna().sum())

Timestamp             0
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64


# Split

In [9]:
# Convert the timestamp column into actual dates
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="raise")

# Arrange transactions from oldest to newest
df = df.sort_values("Timestamp").reset_index(drop=True)

# Find timestamps around the 70% and 85% positions
train_cutoff = df["Timestamp"].iloc[int(len(df) * 0.70)]
test_cutoff = df["Timestamp"].iloc[int(len(df) * 0.85)]

# Oldest ~70%: training
train_df = df[df["Timestamp"] < train_cutoff].copy()

# Next ~15%: validation
val_df = df[
    (df["Timestamp"] >= train_cutoff)
    & (df["Timestamp"] < test_cutoff)
].copy()

# Newest ~15%: testing
test_df = df[df["Timestamp"] >= test_cutoff].copy()

for name, data in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df),
]:
    print(f"\n{name}: {len(data):,} transactions")
    print("From:", data["Timestamp"].min())
    print("To:  ", data["Timestamp"].max())

    # Change this if your label column has a different name
    print(data["Is Laundering"].value_counts())


Training: 3,554,647 transactions
From: 2022-09-01 00:00:00
To:   2022-09-07 14:54:00
Is Laundering
0    3551791
1       2856
Name: count, dtype: int64

Validation: 761,590 transactions
From: 2022-09-07 14:55:00
To:   2022-09-09 03:15:00
Is Laundering
0    760830
1       760
Name: count, dtype: int64

Testing: 762,099 transactions
From: 2022-09-09 03:16:00
To:   2022-09-18 16:18:00
Is Laundering
0    760538
1      1561
Name: count, dtype: int64
